## 0. Setup — imports, paths, hyperparams

In [ ]:
from pathlib import Path
from dataclasses import dataclass
import json
import os

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import functional as TF
from PIL import Image

# ----- paths -----
ROOT = Path.cwd().resolve()
if not (ROOT / "datasets").is_dir() and not (ROOT / "pipeline").is_dir():
    raise RuntimeError("Run from repo root (needs datasets/ or pipeline/).")

DATA_ROOT = ROOT / "datasets"
CKPT_DIR = ROOT / "checkpoints" / "covid"
RESULT_DIR = ROOT / "results" / "covid"
BT_BEST = CKPT_DIR / "bt_encoder_best.pt"
BT_LAST = CKPT_DIR / "bt_encoder_last.pt"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ----- hyperparams -----
RATIO = 0.3
BASE = 32
MU_CH = 32 * BASE

BT_EPOCHS = 200
BT_LR = 3e-4
BT_BATCH = 16

SEG_EPOCHS = 400
SEG_LR = 1e-4
SEG_BATCH = 4
SEG_ES = 40

RECON_WEIGHT = 0.99
BETA_LOCAL = 0.01

CLS_EPOCHS = 40
CLS_ES = 10
CLS_POS_WEIGHT = 2.0
SEED = 0

RUN_BT = False
RUN_SEG = False
RUN_CLS = False


def safe_load(path: Path, map_location=DEVICE):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def atomic_torch_save(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


print("ROOT", ROOT)
print("device", DEVICE)
print("data", DATA_ROOT / "covid_8_1_1")
print("BT", BT_BEST)
print("results", RESULT_DIR)


ROOT C:\Users\Lee\Desktop\An-Empirical-Study-on-BT-VAE-Reproduction-and-Extension
device cpu
data C:\Users\Lee\Desktop\An-Empirical-Study-on-BT-VAE-Reproduction-and-Extension\datasets\covid_8_1_1
BT C:\Users\Lee\Desktop\An-Empirical-Study-on-BT-VAE-Reproduction-and-Extension\checkpoints\covid\bt_encoder_best.pt
results C:\Users\Lee\Desktop\An-Empirical-Study-on-BT-VAE-Reproduction-and-Extension\results\covid


## 1. Data — COVID loader + label-ratio bundle

In [ ]:
class CovidDataset(Dataset):
    """Lazy disk loader. Returns img, mask, view_left(90°), view_right(270°)."""

    def __init__(self, data_dir, img_size=(256, 256)):
        self.data_dir = Path(data_dir)
        self.img_size = tuple(img_size)
        self.path_original = self.data_dir / "original"
        self.path_label = self.data_dir / "label"
        
        if not self.path_original.is_dir():
            raise FileNotFoundError(f"missing original/: {self.path_original}")
        if not self.path_label.is_dir():
            raise FileNotFoundError(f"missing label/: {self.path_label}")
       
        names = sorted(
            p.name for p in self.path_original.iterdir()
            if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
        )
       
        self.filenames = [n for n in names if (self.path_label / n).exists()]
       
        if not self.filenames:
            raise RuntimeError(f"no image/mask pairs in {self.data_dir}")

    def __len__(self):
        return len(self.filenames)

    def _load_gray(self, path: Path) -> torch.Tensor:
        with Image.open(path) as img:
            img = img.convert("L").resize(self.img_size, Image.BILINEAR)
            t = torch.from_numpy(np.asarray(img, dtype=np.float32))
        return t / 255.0

    def _load_mask(self, path: Path) -> torch.Tensor:
        with Image.open(path) as img:
            img = img.convert("L").resize(self.img_size, Image.NEAREST)
            t = torch.from_numpy(np.asarray(img, dtype=np.float32))
        return (t > 0).float()

    def __getitem__(self, idx):

        n = len(self.filenames)
        last_err = None

        for offset in range(n):
            j = (idx + offset) % n
            name = self.filenames[j]

            try:
                image = self._load_gray(self.path_original / name)
                label = self._load_mask(self.path_label / name)

            except (OSError, ValueError, SyntaxError) as e:
                last_err = e
                continue

            left = TF.rotate(image.unsqueeze(0), 90)
            right = TF.rotate(image.unsqueeze(0), 270)

            return image, label, left, right

        raise OSError(f"all images failed near idx={idx}: {last_err}")


@dataclass
class DataBundle:
    name: str
    root: Path
    train: CovidDataset
    val: CovidDataset
    test: CovidDataset
    train_loader: DataLoader
    val_loader: DataLoader
    test_loader: DataLoader


def load_covid(batch_size=16, num_workers=0, pin_memory=False, img_size=(256, 256)) -> DataBundle:
    root = DATA_ROOT / "covid_8_1_1"
    train = CovidDataset(root / "train", img_size=img_size)
    val = CovidDataset(root / "val", img_size=img_size)
    test = CovidDataset(root / "test", img_size=img_size)
    kw = dict(num_workers=num_workers, pin_memory=pin_memory)

    return DataBundle(
        name="covid",
        root=root,
        train=train,
        val=val,
        test=test,
        train_loader=DataLoader(train, batch_size=batch_size, shuffle=True, drop_last=True, **kw),
        val_loader=DataLoader(val, batch_size=batch_size, shuffle=False, drop_last=False, **kw),
        test_loader=DataLoader(test, batch_size=batch_size, shuffle=False, drop_last=False, **kw),
    )


class IndexSubset(Dataset):
    def __init__(self, base, indices):
        self.base = base
        self.indices = list(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        return self.base[self.indices[i]]


def patient_id(name: str) -> str:
    stem = Path(name).stem
    return stem.rsplit("_", 1)[0] if "_" in stem else stem


def subset_indices(names, ratio: float, seed: int = SEED):

    """Patient-stratified subsample; count = round(n * ratio)."""
    groups = {}
    
    for i, name in enumerate(names):
        groups.setdefault(patient_id(name), []).append(i)

    rng = np.random.RandomState(seed)

    target = min(len(names), max(1, int(round(len(names) * ratio))))

    picked, leftover = [], []
    
    for pid in sorted(groups):
        idx = groups[pid][:]
        
        rng.shuffle(idx)

        k = int(np.floor(len(idx) * ratio))
        picked.extend(idx[:k])
        leftover.extend(idx[k:])


    rng.shuffle(leftover)
    
    if len(picked) < target:
        picked.extend(leftover[: target - len(picked)])
    
    elif len(picked) > target:
        rng.shuffle(picked)
        picked = picked[:target]
    
    return sorted(picked)


def make_bundle(ratio: float, batch_size: int):
    full = load_covid(batch_size=batch_size, num_workers=0, pin_memory=False)
   
    idx = subset_indices(list(full.train.filenames), ratio)
    train = IndexSubset(full.train, idx)
   
    bs = min(batch_size, max(1, len(train)))
   
    train_loader = DataLoader(train, batch_size=bs, shuffle=True, num_workers=0, pin_memory=False)
    val_loader = DataLoader(full.val, batch_size=bs, shuffle=False, num_workers=0, pin_memory=False)
    test_loader = DataLoader(full.test, batch_size=bs, shuffle=False, num_workers=0, pin_memory=False)
 
    return full, train_loader, val_loader, test_loader, len(idx)


print("data helpers ready:", CovidDataset.__name__, make_bundle.__name__)


data helpers ready: CovidDataset make_bundle


## 2. Models — Encoder, S5 Decoder, VAE, LesionPresenceHead

In [ ]:
class ResBlockUp(nn.Module):
    def __init__(self, filters_in, filters_out, act=True):
        super().__init__()
        self.act = act
        
        self.conv1_block = nn.Sequential(
            nn.Conv2d(filters_in, filters_in, 3, stride=1, padding=1),
            nn.Upsample(scale_factor=2, mode="nearest"),
            nn.BatchNorm2d(filters_in),
            nn.LeakyReLU(0.2, inplace=True),
        )
        
        self.conv2_block = nn.Sequential(
            nn.Conv2d(filters_in, filters_out, 3, stride=1, padding=1),
            nn.BatchNorm2d(filters_out),
        )
        
        self.conv3_block = nn.Sequential(
            nn.Conv2d(filters_in, filters_out, 3, stride=1, padding=1),
            nn.Upsample(scale_factor=2, mode="nearest"),
            nn.BatchNorm2d(filters_out),
            nn.LeakyReLU(0.2, inplace=True),
        )

        self.lrelu = nn.LeakyReLU(0.2, inplace=False)

    def forward(self, x):
        conv1 = self.conv1_block(x)
        conv2 = self.conv2_block(conv1)

        if self.act:
            conv2 = self.lrelu(conv2)
        
        conv3 = self.conv3_block(x)
        
        if self.act:
            conv3 = self.lrelu(conv3)
        
        return conv2 + conv3


class ResBlockDown(nn.Module):
    def __init__(self, filters_in, filters_out, act=True):
        super().__init__()
        self.act = act
        self.conv1_block = nn.Sequential(
            nn.Conv2d(filters_in, filters_in, 3, stride=2, padding=1),
            nn.BatchNorm2d(filters_in),
            nn.LeakyReLU(0.2, inplace=True),
        )

        self.conv2_block = nn.Sequential(
            nn.Conv2d(filters_in, filters_out, 3, stride=1, padding=1),
            nn.BatchNorm2d(filters_out),
        )
        
        self.conv3_block = nn.Sequential(
            nn.Conv2d(filters_in, filters_out, 3, stride=2, padding=1),
            nn.BatchNorm2d(filters_out),
        )
        
        self.lrelu = nn.LeakyReLU(0.2, inplace=False)

    def forward(self, x):
        conv1 = self.conv1_block(x)
        conv2 = self.conv2_block(conv1)
        
        if self.act:
            conv2 = self.lrelu(conv2)
        
        conv3 = self.conv3_block(x)
        
        if self.act:
            conv3 = self.lrelu(conv3)
        
        return conv2 + conv3


class ConcatConvFuse(nn.Module):
    """cat(decoder, skip) -> 3x3 Conv -> BN -> LeakyReLU."""

    def __init__(self, dec_ch: int, skip_ch: int):
        super().__init__()

        self.fuse = nn.Sequential(
            nn.Conv2d(dec_ch + skip_ch, dec_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(dec_ch),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        
        if skip.shape[-2:] != x.shape[-2:]:
            skip = F.interpolate(skip, size=x.shape[-2:], mode="bilinear", align_corners=False)
        
        return self.fuse(torch.cat([x, skip], dim=1))


class Encoder(nn.Module):
    """Same weights layout as original.models1.Encoder; forward also returns skips."""

    def __init__(self, base, dropout_rate=0.1):
        super().__init__()
        self.encoder_initial_flair = self._create_initial_encoder(1, base, dropout_rate)
        self.encoder_features_flair = self._create_feature_encoder(base, dropout_rate)
        self.encoder_mu_flair = self._create_mu_encoder(32 * base)
        self.encoder_logvar_flair = self._create_logvar_encoder(32 * base)

    def _create_initial_encoder(self, in_channels, base, dropout_rate):
        return nn.Sequential(
            nn.Conv2d(in_channels, base, 3, stride=1, padding=1),
            nn.BatchNorm2d(base),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout2d(p=dropout_rate),
        )

    def _create_feature_encoder(self, base, dropout_rate):
        return nn.Sequential(
            ResBlockDown(base, 2 * base),
            nn.Dropout2d(p=dropout_rate),
            ResBlockDown(2 * base, 4 * base),
            nn.Dropout2d(p=dropout_rate),
            ResBlockDown(4 * base, 8 * base),
            nn.Dropout2d(p=dropout_rate),
            ResBlockDown(8 * base, 16 * base),
            nn.Dropout2d(p=dropout_rate),
            ResBlockDown(16 * base, 32 * base),
            nn.Dropout2d(p=dropout_rate),
        )

    def _create_mu_encoder(self, C):
        return nn.Sequential(nn.Conv2d(C, C, 1), nn.BatchNorm2d(C))

    def _create_logvar_encoder(self, C):
        return nn.Sequential(nn.Conv2d(C, C, 1), nn.BatchNorm2d(C))

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward_features(self, flair):
        e0 = self.encoder_initial_flair(flair)
        e1 = self.encoder_features_flair[0](e0)
        e1d = self.encoder_features_flair[1](e1)
        e2 = self.encoder_features_flair[2](e1d)
        e2d = self.encoder_features_flair[3](e2)
        e3 = self.encoder_features_flair[4](e2d)
        e3d = self.encoder_features_flair[5](e3)
        e4 = self.encoder_features_flair[6](e3d)
        e4d = self.encoder_features_flair[7](e4)
        h = self.encoder_features_flair[8](e4d)
        h = self.encoder_features_flair[9](h)
        mu = self.encoder_mu_flair(h)
        logvar = self.encoder_logvar_flair(h)
        z = self.reparameterize(mu, logvar)
        return mu, logvar, z, (e0, e1, e2, e3, e4)

    def forward(self, flair):
        mu, logvar, z, _ = self.forward_features(flair)
        return mu, logvar, z


class Decoder(nn.Module):
    """S5: B0..B3 concat+3x3 fuse, B4 off."""

    def __init__(self, base, dropout_rate=0.1):
        super().__init__()
        feat = 32 * base
        self.up1 = ResBlockUp(feat, feat)
        self.drop1 = nn.Dropout2d(p=dropout_rate)
        self.up2 = ResBlockUp(feat, feat)
        self.drop2 = nn.Dropout2d(p=dropout_rate)
        self.up3 = ResBlockUp(feat, 16 * base)
        self.drop3 = nn.Dropout2d(p=dropout_rate)
        self.up4 = ResBlockUp(16 * base, 8 * base)
        self.drop4 = nn.Dropout2d(p=dropout_rate)
        self.up5 = ResBlockUp(8 * base, base)
        self.drop5 = nn.Dropout2d(p=dropout_rate)
        self.out_conv = nn.Conv2d(base, 1, 3, stride=1, padding=1)

        self.fuse3 = ConcatConvFuse(feat, 8 * base)  # after up2 + e3
        self.fuse2 = ConcatConvFuse(16 * base, 4 * base)
        self.fuse1 = ConcatConvFuse(8 * base, 2 * base)
        self.fuse0 = ConcatConvFuse(base, base)

    def forward(self, z, skips):
        e0, e1, e2, e3, e4 = skips  # e4 unused (B4 off)
        del e4

        x = self.drop1(self.up1(z))
        x = self.drop2(self.up2(x))
        x = self.fuse3(x, e3)
        x = self.drop3(self.up3(x))
        x = self.fuse2(x, e2)
        x = self.drop4(self.up4(x))
        x = self.fuse1(x, e1)
        x = self.drop5(self.up5(x))
        x = self.fuse0(x, e0)
        return self.out_conv(x)


class VAE(nn.Module):
    def __init__(self, base, dropout_rate=0.1):
        super().__init__()
        self.encoder = Encoder(base, dropout_rate)
        self.decoder = Decoder(base, dropout_rate)

    def forward(self, x):
        mu, logvar, z, skips = self.encoder.forward_features(x)
        out = self.decoder(z, skips)
        return out, mu, logvar


class LesionPresenceHead(nn.Module):
    """Case-level presence logit from spatial μ. No sigmoid inside."""

    def __init__(self, in_channels: int, hidden_dim: int = 128, dropout: float = 0.5):
        super().__init__()
        self.in_channels = int(in_channels)
        self.hidden_dim = int(hidden_dim)
        self.dropout = float(dropout)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.classifier = nn.Sequential(
            nn.Linear(in_channels * 2, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )


    def forward(self, feature: torch.Tensor) -> torch.Tensor:
        avg_feature = self.avg_pool(feature).flatten(1)
        max_feature = self.max_pool(feature).flatten(1)
        pooled = torch.cat([avg_feature, max_feature], dim=1)
        logits = self.classifier(pooled)
        return logits.squeeze(1)


## 3. BT helpers — loss, freeze BN, load checkpoint

In [ ]:
def kl_gaussian(mu1, sigma1, mu2, sigma2, eps: float = 1e-12):
    var1 = (sigma1 ** 2).clamp_min(eps)
    var2 = (sigma2 ** 2).clamp_min(eps)
    diff2 = (mu1 - mu2) ** 2
    return 0.5 * (torch.log(var2) - torch.log(var1) + (var1 + diff2) / var2 - 1.0)


def kl_matrix_loss_weighted(K, w_diag=1.0, w_off=1.0, target_off=3.0):

    eye = torch.eye(K.size(0), device=K.device, dtype=K.dtype).bool()
    diag_loss = (K.diagonal() ** 2).mean()
    off_loss = ((K[~eye] - target_off) ** 2).mean()

    return w_diag * diag_loss + w_off * off_loss


def cross_corr_mu_loss(m1v, m2v, eps=1e-12, lambda_off=1.0):
    B = m1v.size(0)

    z1n = (m1v - m1v.mean(0)) / (m1v.std(0) + eps)
    z2n = (m2v - m2v.mean(0)) / (m2v.std(0) + eps)
   
    C = (z1n.T @ z2n) / float(B)
    D = C.size(0)
   
    diag = torch.diag(C)
    off = C - torch.diag(diag)
   
    diag_loss = (1.0 - diag).pow(2).mean()
    off_loss = off.pow(2).sum() / (D * (D - 1) + eps)
   
    return diag_loss + lambda_off * off_loss, C


def bt_batch_loss(encoder, view1, view2, ep: int):
    
    if view1.ndim == 3:
        view1 = view1.unsqueeze(1)
    
    if view2.ndim == 3:
        view2 = view2.unsqueeze(1)

    m1, v1_, _z1 = encoder(view1)
    m2, v2_, _z2 = encoder(view2)
    
    v1_ = v1_.clamp(-5, 2)
    v2_ = v2_.clamp(-5, 2)
    
    gap = nn.AdaptiveAvgPool2d(1)

    def pool(x):
        return gap(x).flatten(1) if x.dim() == 4 else x

    m1v, v1v = pool(m1), pool(v1_)
    m2v, v2v = pool(m2), pool(v2_)
    
    sig1 = (0.5 * v1v).exp()
    sig2 = (0.5 * v2v).exp()
    
    K12 = kl_gaussian(m1v.unsqueeze(2), sig1.unsqueeze(2), m2v.unsqueeze(1), sig2.unsqueeze(1))
    K21 = kl_gaussian(m2v.unsqueeze(2), sig2.unsqueeze(2), m1v.unsqueeze(1), sig1.unsqueeze(1))
    
    K = 0.5 * (K12 + K21).mean(dim=0)
    
    tgt = 1.0 + min(1.0, ep / 100.0)
    
    loss_kl = kl_matrix_loss_weighted(K, w_diag=1.0, w_off=1.0, target_off=tgt)
    loss_cmu, _ = cross_corr_mu_loss(m1v, m2v, lambda_off=1.0)
    
    return loss_kl + 0.2 * loss_cmu


def freeze_bn(encoder: nn.Module):
    for m in encoder.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()
            for p in m.parameters():
                p.requires_grad = False


def load_bt_into_encoder(encoder: nn.Module, bt_ckpt: Path):
    st = safe_load(bt_ckpt, map_location="cpu")
    
    enc_state = st["encoder"] if isinstance(st, dict) and "encoder" in st else st
    
    enc_state = {
        (k[len("module."):] if k.startswith("module.") else k): v
        for k, v in enc_state.items()
    }
    
    missing, unexpected = encoder.load_state_dict(enc_state, strict=False)
    
    print(f"encoder load missing={len(missing)} unexpected={len(unexpected)}")


print("BT helpers ready")


BT helpers ready


## 4. Seg loss + metrics

In [ ]:
def binary_dice_loss_sigmoid(pred, target, smooth=1e-6):
    
    pred = torch.sigmoid(pred)
   
    pred_flat = pred.reshape(pred.size(0), -1)
    target_flat = target.reshape(target.size(0), -1)
   
    intersection = (pred_flat * target_flat).sum(dim=1)

    union = pred_flat.sum(dim=1) + target_flat.sum(dim=1)
    
    dice = (2.0 * intersection + smooth) / (union + smooth)
    
    return 1 - dice.mean()


def combined_binary_loss_sigmoid(pred, target, dice_weight=0.5, ce_weight=0.5):
   
    dice = binary_dice_loss_sigmoid(pred, target)
    ce = nn.BCEWithLogitsLoss()(pred.view(pred.size(0), -1), target.view(target.size(0), -1))
   
    return dice_weight * dice + ce_weight * ce, float(dice.item()), float(ce.item())


def seg_step_loss(model, flair, seg, return_out=False):

    if flair.ndim == 3:
        flair = flair.unsqueeze(1)
   
    if seg.ndim == 3:
        seg = seg.unsqueeze(1)
   
    output, mu_local, logvar_local = model(flair)
   
    dice_loss_combined, _, _ = combined_binary_loss_sigmoid(output, seg)
   
    kl = 1 + logvar_local - torch.square(mu_local) - torch.exp(logvar_local)
    kl = -0.5 * torch.sum(kl, dim=-1)
   
    loss = RECON_WEIGHT * dice_loss_combined + BETA_LOCAL * torch.mean(kl)
    stats = (loss, float(dice_loss_combined.item()), float(torch.mean(kl).item()))
   
    if return_out:
        return stats + (output, seg)
   
    return stats


@torch.no_grad()
def batch_dice(logits, target, thr=0.5):

    pred = (torch.sigmoid(logits) > thr).float()
   
    if target.ndim == 3:
        target = target.unsqueeze(1)

    inter = (pred * target).sum(dim=(1, 2, 3))
    union = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))

    return ((2 * inter + 1e-6) / (union + 1e-6)).detach().cpu().numpy()


@torch.no_grad()
def batch_pred_gt_sums(logits, target, thr=0.5):

    pred = (torch.sigmoid(logits) > thr).float()
    
    if target.ndim == 3:
        target = target.unsqueeze(1)
    
    return (
        pred.sum(dim=(1, 2, 3)).detach().cpu().numpy(),
        target.sum(dim=(1, 2, 3)).detach().cpu().numpy(),
    )


@torch.no_grad()
def split_dice_stats(dices, gt_sums, pred_sums):
    
    nonempty = gt_sums > 0
    empty = ~nonempty
    ne = dices[nonempty]
   
    return {
        "dice_mean": float(dices.mean()),
        "dice_median": float(np.median(dices)),

        "mean_dice_nonempty": float(ne.mean()) if ne.size else None,
        "median_dice_nonempty": float(np.median(ne)) if ne.size else None,
      
        "n": int(dices.size),
        "n_nonempty": int(nonempty.sum()),
        "n_empty": int(empty.sum()),
      
        "empty_fp_rate": float((pred_sums[empty] > 0).mean()) if empty.any() else None,
    }


def binary_auroc(y: np.ndarray, s: np.ndarray) -> float:
 
    y = y.astype(bool)
    n_pos, n_neg = int(y.sum()), int((~y).sum())
 
    if n_pos == 0 or n_neg == 0:
        return 0.5
 
    order = np.argsort(s)
    ranks = np.empty(len(s), dtype=np.float64)
    ranks[order] = np.arange(1, len(s) + 1, dtype=np.float64)
 
    return float((ranks[y].sum() - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg))


def conf_matrix(y: np.ndarray, pred: np.ndarray) -> dict:
   
    y = y.astype(bool)
    pred = pred.astype(bool)
   
    tt = int((y & pred).sum())
    ft = int((~y & pred).sum())
    tf = int((y & ~pred).sum())
    ff = int((~y & ~pred).sum())
   
    sens = tt / max(1, tt + tf)
    spec = ff / max(1, ff + ft)
   
    return {
        "TT": tt, "FT": ft, "TF": tf, "FF": ff,
        "sensitivity": float(sens),
        "specificity": float(spec),
        "precision": float(tt / max(1, tt + ft)),
        "npv": float(ff / max(1, ff + tf)),
        "balanced_accuracy": float(0.5 * (sens + spec)),
        "empty_fp_rate": float(ft / max(1, ft + ff)),
    }


def apply_case_rejection(dices_raw, pred_mass, gt, keep_mask: np.ndarray):
   
    pred = pred_mass.copy()
    pred[~keep_mask] = 0
   
    dices = dices_raw.copy()
    rejected = ~keep_mask
   
    empty = gt == 0
   
    dices[rejected & empty] = 1.0
    dices[rejected & ~empty] = 0.0
   
    return split_dice_stats(dices, gt, pred), pred


def metrics_bundle(gt, pred_mass, dices, keep: np.ndarray) -> dict:
    
    st, pred = apply_case_rejection(dices, pred_mass, gt, keep)
   
    cm = conf_matrix(gt > 0, pred > 0)
   
    return {
        "overall_dice": float(st["dice_mean"]),
        "nonempty_dice": float(st["mean_dice_nonempty"]),
        "empty_fp_rate": float(st["empty_fp_rate"]),
        **cm,
    }


print("seg/metrics ready")


seg/metrics ready


## 5. CLS helpers — collect scores, pick τ, eval S5


In [ ]:
def prep_xy(batch):
    x, y = batch[0].to(DEVICE), batch[1].to(DEVICE)
   
    if x.ndim == 3:
        x = x.unsqueeze(1)
   
    if y.ndim == 3:
        y = y.unsqueeze(1)
   
    return x, y


@torch.inference_mode()
def eval_split(model, loader, kind: str = "s5") -> dict:
   
    model.eval()
    dices, gts, preds = [], [], []
   
    for batch in loader:
        x, y = prep_xy(batch)
   
        if kind == "s5":
            mu, _lv, _z, skips = model.encoder.forward_features(x)
            logits = model.decoder(mu, skips)
        else:
            mu, _lv, _z = model.encoder(x)
            logits = model.decoder(mu)
   
        dices.append(batch_dice(logits, y))
        ps, gs = batch_pred_gt_sums(logits, y)
        preds.append(ps)
        gts.append(gs)
  
    st = split_dice_stats(np.concatenate(dices), np.concatenate(gts), np.concatenate(preds))
    
    return {
        "dice": st["dice_mean"],
        "dice_median": st["dice_median"],
        "nonempty_dice": st["mean_dice_nonempty"],
        "empty_fp_rate": st["empty_fp_rate"],
        "n": st["n"],
    }


@torch.inference_mode()
def collect_s5_cls(model, head, loader):
    model.eval()
    head.eval()
    bags = {"gt": [], "mass": [], "dice": [], "score": []}
    for batch in loader:
        x, y = prep_xy(batch)
        mu, _lv, _z, skips = model.encoder.forward_features(x)
        logits = model.decoder(mu, skips)
        ps, gs = batch_pred_gt_sums(logits, y)
        bags["gt"].append(gs)
        bags["mass"].append(ps)
        bags["dice"].append(batch_dice(logits, y))
        bags["score"].append(torch.sigmoid(head(mu)).detach().cpu().numpy())
    return {k: np.concatenate(v) for k, v in bags.items()}


def pick_tau(val):
    best = None
    for i in range(1, 100):
        thr = i / 100.0
        m = metrics_bundle(val["gt"], val["mass"], val["dice"], val["score"] >= thr)
        key = (m["overall_dice"], -m["empty_fp_rate"])
        if best is None or key > best["key"]:
            best = {"thr": thr, "key": key, "m": m}
    return best["thr"], best["m"]


print("CLS helpers ready")


CLS helpers ready


## 6. Train — Barlow Twins


In [8]:
def train_bt():
    CKPT_DIR.mkdir(parents=True, exist_ok=True)
    if BT_BEST.exists():
        print("skip BT", BT_BEST)
        return BT_BEST

    data = load_covid(batch_size=BT_BATCH, num_workers=0, pin_memory=False)
    encoder = Encoder(base=BASE).to(DEVICE)
    freeze_bn(encoder)
    opt = torch.optim.AdamW(encoder.parameters(), lr=BT_LR, weight_decay=1e-4)
    best_val = float("inf")

    for ep in range(1, BT_EPOCHS + 1):
        encoder.train()
        freeze_bn(encoder)
        running = 0.0
        for batch in data.train_loader:
            view1, view2 = batch[2].to(DEVICE), batch[3].to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss = bt_batch_loss(encoder, view1, view2, ep)
            loss.backward()
            nn.utils.clip_grad_norm_(encoder.parameters(), 1.0)
            opt.step()
            running += float(loss.item())

        encoder.eval()
        val_running = 0.0
        with torch.no_grad():
            for batch in data.val_loader:
                view1, view2 = batch[2].to(DEVICE), batch[3].to(DEVICE)
                val_running += float(bt_batch_loss(encoder, view1, view2, ep).item())

        train_avg = running / max(1, len(data.train_loader))
        val_avg = val_running / max(1, len(data.val_loader))
        print(f"[BT] ep {ep}/{BT_EPOCHS} train={train_avg:.6f} val={val_avg:.6f}")

        payload = {"ep": ep, "encoder": encoder.state_dict(), "best_val": best_val}
        atomic_torch_save(payload, BT_LAST)
        if val_avg < best_val - 1e-6:
            best_val = val_avg
            payload["best_val"] = best_val
            atomic_torch_save(payload, BT_BEST)

    out = BT_BEST if BT_BEST.exists() else BT_LAST
    print("BT done", out)
    return out


if RUN_BT:
    train_bt()


## 7. Train — S5 segmentation

BT 인코더로 초기화 후 encoder+decoder 학습.  
Early stop: val loss 40 epoch 정체. 테스트는 μ 디코드.



In [9]:
def train_s5(ratio=RATIO):
    pct = int(round(ratio * 100))
    out_dir = RESULT_DIR / "label_ratio" / f"r{pct:02d}" / "s5"
    out_dir.mkdir(parents=True, exist_ok=True)
    best_path = out_dir / "seg_best.pt"
    last_path = out_dir / "seg_last.pt"
    report_path = out_dir / "report_seg.json"
    if best_path.exists() and report_path.exists():
        print("skip SEG", best_path)
        return best_path

    _full, train_loader, val_loader, test_loader, n_train = make_bundle(ratio, SEG_BATCH)
    model = VAE(base=BASE).to(DEVICE)
    bt = BT_BEST if BT_BEST.exists() else BT_LAST
    if not last_path.exists():
        if not bt.exists():
            raise FileNotFoundError(f"BT checkpoint missing: {bt}. Set RUN_BT=True first.")
        load_bt_into_encoder(model.encoder, bt)

    opt = torch.optim.AdamW(model.parameters(), lr=SEG_LR, weight_decay=1e-4)
    sched = ReduceLROnPlateau(opt, mode="min", factor=0.95, patience=3, min_lr=1e-8)
    best_loss, best_ep, bad = float("inf"), 0, 0
    start_ep = 1
    if last_path.exists():
        st = safe_load(last_path, map_location="cpu")
        model.load_state_dict(st["model"])
        start_ep = int(st.get("ep", 0)) + 1
        best_loss = float(st.get("best_loss", best_loss))
        best_ep = int(st.get("best_ep", 0))
        bad = max(0, start_ep - 1 - best_ep)
        print(f"resume S5 from ep {start_ep}")

    for ep in range(start_ep, SEG_EPOCHS + 1):
        model.train()
        tr_loss, ntr = 0.0, 0
        for batch in train_loader:
            x, y = batch[0].to(DEVICE), batch[1].to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss, _dice, _kl = seg_step_loss(model, x, y)
            loss.backward()
            opt.step()
            tr_loss += float(loss.item())
            ntr += 1

        model.eval()
        va_loss, nva = 0.0, 0
        with torch.no_grad():
            for batch in val_loader:
                x, y = batch[0].to(DEVICE), batch[1].to(DEVICE)
                loss, _, _ = seg_step_loss(model, x, y)
                va_loss += float(loss.item())
                nva += 1
        va_loss /= max(1, nva)
        sched.step(va_loss)
        print(f"[S5 r{pct} {ep}/{SEG_EPOCHS}] train={tr_loss / max(1, ntr):.4f} val={va_loss:.4f}")

        payload = {
            "model": model.state_dict(),
            "ep": ep,
            "best_loss": best_loss,
            "best_ep": best_ep,
            "ratio": ratio,
            "mode": "s5",
        }
        atomic_torch_save(payload, last_path)
        if va_loss < best_loss - 1e-5:
            best_loss, best_ep, bad = va_loss, ep, 0
            payload["best_loss"] = best_loss
            payload["best_ep"] = best_ep
            atomic_torch_save(payload, best_path)
        else:
            bad += 1
            if bad >= SEG_ES:
                print(f"early stop S5 @ {ep}")
                break

    ckpt = best_path if best_path.exists() else last_path
    model.load_state_dict(safe_load(ckpt, map_location="cpu")["model"])
    test = eval_split(model, test_loader, "s5")
    report = {
        "ratio": ratio,
        "mode": "s5",
        "n_train": n_train,
        "best_ep": best_ep,
        "best_val_loss": best_loss,
        "test": test,
    }
    report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(report)
    return ckpt


if RUN_SEG:
    train_s5(RATIO)


## 8. Train — lesion-presence head + τ gate

S5 고정. Head(μ)만 학습. 검증 Dice 최대 τ로 테스트 마스크 게이트.



In [10]:
def train_s5_cls(ratio=RATIO):
    pct = int(round(ratio * 100))
    seg_ckpt = RESULT_DIR / "label_ratio" / f"r{pct:02d}" / "s5" / "seg_best.pt"
    out_dir = RESULT_DIR / "label_ratio" / f"r{pct:02d}" / "s5_cls"
    out_dir.mkdir(parents=True, exist_ok=True)
    report_path = out_dir / "report.json"
    if report_path.exists():
        print("skip CLS", report_path)
        return json.loads(report_path.read_text(encoding="utf-8"))
    if not seg_ckpt.exists():
        raise FileNotFoundError(f"S5 checkpoint missing: {seg_ckpt}. Set RUN_SEG=True first.")

    _full, train_loader, val_loader, test_loader, n_train = make_bundle(ratio, SEG_BATCH)
    model = VAE(base=BASE).to(DEVICE)
    model.load_state_dict(safe_load(seg_ckpt, map_location="cpu")["model"])
    model.eval()
    for p in model.parameters():
        p.requires_grad = False

    head = LesionPresenceHead(MU_CH).to(DEVICE)
    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([CLS_POS_WEIGHT], device=DEVICE))
    opt = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-4)
    best_auroc, bad, best_ep = -1.0, 0, 0
    best_path = out_dir / "head_best.pt"
    trend = []

    for ep in range(1, CLS_EPOCHS + 1):
        head.train()
        total, n = 0.0, 0
        for batch in train_loader:
            x, y = prep_xy(batch)
            with torch.no_grad():
                mu, _lv, _z, _sk = model.encoder.forward_features(x)
            y_cls = (y.flatten(1).sum(1) > 0).float()
            opt.zero_grad(set_to_none=True)
            loss = bce(head(mu), y_cls)
            loss.backward()
            opt.step()
            total += float(loss.item()) * y_cls.size(0)
            n += int(y_cls.size(0))

        val = collect_s5_cls(model, head, val_loader)
        auroc = binary_auroc(val["gt"] > 0, val["score"])
        trend.append({"ep": ep, "train_bce": total / max(n, 1), "val_auroc": auroc})
        print(f"[CLS] ep {ep} bce={trend[-1]['train_bce']:.4f} auroc={auroc:.4f}")

        if auroc > best_auroc + 1e-4:
            best_auroc, bad, best_ep = auroc, 0, ep
            atomic_torch_save({"head": head.state_dict(), "ep": ep, "auroc": auroc}, best_path)
        else:
            bad += 1
            if bad >= CLS_ES:
                break

    if not best_path.exists():
        raise RuntimeError("CLS head never improved on validation AUROC.")

    head.load_state_dict(safe_load(best_path, map_location="cpu")["head"])
    val = collect_s5_cls(model, head, val_loader)
    test = collect_s5_cls(model, head, test_loader)
    tau, _ = pick_tau(val)
    mt = metrics_bundle(test["gt"], test["mass"], test["dice"], test["score"] >= tau)
    report = {
        "ratio": ratio,
        "n_train": n_train,
        "loss": "BCEWithLogits",
        "probability": "p = sigmoid(Head(mu)) = P(lesion present | slice)",
        "tau": tau,
        "best_ep": best_ep,
        "val_auroc": best_auroc,
        "test_dice": float(mt["overall_dice"]),
        "test_nonempty_dice": float(mt["nonempty_dice"]),
        "test_empty_fp_rate": float(mt["empty_fp_rate"]),
    }
    (out_dir / "prob_trend.json").write_text(json.dumps(trend, indent=2), encoding="utf-8")
    report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(report)
    return report


if RUN_CLS:
    train_s5_cls(RATIO)
